In [ ]:
import numpy as np # linear algebra
import pandas as pd
import os
import itertools
from keras.layers import Attention
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv1D, GlobalAveragePooling1D, BatchNormalization, Bidirectional, GRU, Dense, Dropout
from tensorflow.keras.utils import to_categorical
import psutil
import os
from imblearn.over_sampling import BorderlineSMOTE,SMOTE
from collections import Counter
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    matthews_corrcoef
)

import time
from sklearn.metrics import classification_report, confusion_matrix,precision_recall_fscore_support, roc_auc_score
from imblearn.metrics import geometric_mean_score



In [ ]:

# Function to report memory (MB)
def memory_usage_mb():
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 ** 2)

# Initial memory
mem_start = memory_usage_mb()
print(f"Initial memory usage: {mem_start:.2f} MB")
# =====================================================
# 1. Load and combine CSV files
# =====================================================
# Load and Preprocess Data
# ----------------------
csv_dir = "IoT-23"
dfs = []



filename = f"{csv_dir}/iot23_combined_new.csv"

try:
    print(f"Loading: {filename}")
    df_combined = df_combined = pd.read_csv(filename)
    print("DataFrame shape:", df_combined.shape)
except FileNotFoundError:
    raise FileNotFoundError(f"File not found: {filename}")
mem_after_data = memory_usage_mb()
print(f"Memory after data load: {mem_after_data:.2f} MB")


In [ ]:
label = 'label'

In [ ]:
print("Original Classes")
class_counts = df_combined[label].value_counts().reset_index()
class_counts.columns = ['Class', 'Count']

display(class_counts)

In [ ]:

# Create binary labels
df_combined["label"] = df_combined["label"].apply(
    lambda x: "Benign" if x == "Benign" else "Attack"
)


In [ ]:
print("Original Binary Classes")
class_counts = df_combined[label].value_counts().reset_index()
class_counts.columns = ['Class', 'Count']

display(class_counts)

In [ ]:
print("No of Samples before:")
print(df_combined.shape[0])

In [ ]:
import pandas as pd
from IPython.display import display
print("ORIGINAL FEATURE INFO")

feature_info = pd.DataFrame({
    'Feature': df_combined.columns,
    'Data Type': df_combined.dtypes.astype(str),
    'Missing': df_combined.isna().sum(),
    'Missing %': (df_combined.isna().mean() * 100).round(2),
    'Unique': df_combined.nunique(),
    'Mean': df_combined.mean(numeric_only=True),
    'Std': df_combined.std(numeric_only=True),
    'Min': df_combined.min(numeric_only=True),
    'Max': df_combined.max(numeric_only=True)
})

display(feature_info.round(4))

In [ ]:
print("Total NaNs in dataset: ", df_combined.isna().sum().sum())

In [ ]:
df_combined.head()

In [ ]:

# =====================================================
# 2. Drop irrelevant / identifier columns
# =====================================================
drop_cols = [
    "ts", "uid" ,"Unnamed: 0"       # hardware identifiers
]
df_combined.drop(columns=drop_cols, inplace=True, errors="ignore")
df_combined[label] = df_combined[label].str.replace("PartOfA", "", regex=False)


In [ ]:
df_combined[label] = df_combined[label].str.strip()

In [ ]:
df_combined = df_combined.rename(columns={ label: 'label'})


In [ ]:
print("Retained Classes")
class_counts = df_combined['label'].value_counts().reset_index()
class_counts.columns = ['Class', 'Count']

display(class_counts)

In [ ]:
retained_features = pd.DataFrame({
    "Feature": df_combined.columns
})

print("Retained Features")
display(retained_features)

In [ ]:
print("No of Samples After:")
print(df_combined.shape[0])

In [ ]:

# Encode labels
print("Encoding labels...")
label_encoder = LabelEncoder()
df_combined['label'] = label_encoder.fit_transform(df_combined['label'])

class_mapping = {
    cls: int(i)
    for i, cls in enumerate(label_encoder.classes_)
}

mapping_df = pd.DataFrame(
    list(class_mapping.items()),
    columns=['Original Label', 'Encoded Label']
)

print("Label Mapping")
display(mapping_df)

In [ ]:
numeric_columns = df_combined.select_dtypes(include=["number"]).columns
df_combined[numeric_columns] = df_combined[numeric_columns].fillna(df_combined[numeric_columns].median())

# Encode categorical variables
non_numeric_columns = df_combined.select_dtypes(exclude=["number"]).columns
for col in non_numeric_columns:
    df_combined[col] = df_combined[col].astype(str)
    le = LabelEncoder()
    df_combined[col] = le.fit_transform(df_combined[col])

In [ ]:
# Separate features and target
X = df_combined.drop(columns=['label'])
y = df_combined['label']

# Define number of classes
n_classes = len(np.unique(y))

# Split the data into training, validation, and test sets
print("Splitting data into training, validation, and test sets...")
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

ST=StandardScaler()

X_train_scaled=ST.fit_transform(X_train)
X_test_scaled=ST.transform(X_test)
X_val_scaled=ST.transform(X_val)

split_labels = (
    ['Train'] * len(y_train) +
    ['Validation'] * len(y_val) +
    ['Test'] * len(y_test)
)
y_combined = pd.Series(list(y_train) + list(y_val) + list(y_test), name='Class')
split_df = pd.DataFrame({'Split': split_labels, 'Class': y_combined})

# Final shapes
print(f"Training set shape: X_train: {X_train_scaled.shape}, y_train: {y_train.shape}")
print(f"Validation set shape: X_val: {X_val_scaled.shape}, y_val: {y_val.shape}")
print(f"Test set shape: X_test: {X_test_scaled.shape}, y_test: {y_test.shape}")


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Get class distribution from df_combined
after_df = (
    y_train
    .value_counts()
    .sort_index()               # Keeps labels in numerical order (0,1,2,...)
    .reset_index()
)

after_df.columns = ['Attack_type', 'Count']

# Plot
plt.figure(figsize=(8, 5))

sns.barplot(
    data=after_df,
    x='Attack_type',
    y='Count',
    palette='tab10'   # Automatically assigns different colors
)

plt.title('Class Distribution before SMOTE')
plt.xlabel('Attack Type')
plt.ylabel('Count')

# Add value labels
for i, value in enumerate(after_df['Count']):
    plt.text(
        i,
        value + after_df['Count'].max() * 0.01,
        f'{value:,}',
        ha='center',
        fontsize=10
    )

plt.tight_layout()
plt.show()

In [ ]:
def scale_dataset(X_train,X_val,X_test,y_train,y_val,y_test):
    # =====================================================
    # 7. Feature scaling
    # =====================================================
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled   = scaler.transform(X_val)
    X_test_scaled  = scaler.transform(X_test)



    y_train_labels = y_train
    orig_counts = Counter(y_train_labels)

    target_count = 800000
    sampling_strategy = {
        cls: target_count
        for cls, cnt in orig_counts.items()
        if cnt < target_count
    }
    print(sampling_strategy)

    # Apply SMOTE to the preprocessed data (X_train_processed)
    sm = SMOTE(random_state=16, sampling_strategy=sampling_strategy)
    X_train_scaled, y_train = sm.fit_resample(X_train_scaled, y_train_labels)


   

    # =====================================================
    # 9. Split consistency check
    # =====================================================
    split_labels = (
        ["Train"] * len(y_train) +
        ["Validation"] * len(y_val) +
        ["Test"] * len(y_test)
    )
    y_combined = pd.Series(list(y_train) + list(y_val) + list(y_test), name="Class")
    split_df = pd.DataFrame({"Split": split_labels, "Class": y_combined})

    return X_train_scaled,X_val_scaled,X_test_scaled,y_train

X_train_scaled,X_val_scaled,X_test_scaled,y_train = scale_dataset(X_train,X_val,X_test,y_train,y_val,y_test)
# Reshape input data for LSTM
print("Reshaping input data for LSTM...")
X_train_reshaped = X_train_scaled.reshape((X_train_scaled.shape[0], X_train_scaled.shape[1], 1))
X_val_reshaped = X_val_scaled.reshape((X_val_scaled.shape[0], X_val_scaled.shape[1], 1))
X_test_reshaped = X_test_scaled.reshape((X_test_scaled.shape[0], X_test_scaled.shape[1], 1))

# One-hot encode the labels
print("One-hot encoding labels...")
y_train_multi_encoded = to_categorical(y_train, num_classes=n_classes)
y_val_multi_encoded = to_categorical(y_val, num_classes=n_classes)
y_test_multi_encoded = to_categorical(y_test, num_classes=n_classes)

# Define input shape
input_shape = X_train_reshaped.shape[1:]
print("Data preprocessing complete.")

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Get class distribution from df_combined
after_df = (
    y_train
    .value_counts()
    .sort_index()               # Keeps labels in numerical order (0,1,2,...)
    .reset_index()
)

after_df.columns = ['Attack_type', 'Count']

# Plot
plt.figure(figsize=(8, 5))

sns.barplot(
    data=after_df,
    x='Attack_type',
    y='Count',
    palette='tab10'   # Automatically assigns different colors
)

plt.title('Class Distribution after SMOTE')
plt.xlabel('Attack Type')
plt.ylabel('Count')

# Add value labels
for i, value in enumerate(after_df['Count']):
    plt.text(
        i,
        value + after_df['Count'].max() * 0.01,
        f'{value:,}',
        ha='center',
        fontsize=10
    )

plt.tight_layout()
plt.show()

In [ ]:
from tensorflow.keras.layers import Input, Bidirectional, GRU, GlobalAveragePooling1D, Dense, Dropout
from tensorflow.keras.models import Model

from tensorflow.keras.layers import Input, LSTM, Bidirectional, Attention
from tensorflow.keras.layers import Dense, BatchNormalization, Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2

def create_model(input_shape, lstm_units, l2_reg,
                         dense1, dense2, n_classes):

    inputs = Input(shape=input_shape)

    x = Bidirectional(
        LSTM(
            lstm_units,
            return_sequences=True,
            kernel_regularizer=l2(l2_reg)
        )
    )(inputs)

    x = Attention()([x, x])

    x = Flatten()(x)

    x = Dense(dense1, activation="relu")(x)
    x = BatchNormalization()(x)

    x = Dense(dense2, activation="relu")(x)

    outputs = Dense(n_classes, activation="softmax")(x)

    model = Model(inputs, outputs)

    return model




In [ ]:

def model_complexity(model, input_shape):
    """
    Computes model complexity metrics.

    Parameters
    ----------
    model : tf.keras.Model
        Trained or untrained Keras model.
    input_shape : tuple
        Shape of one input sample (excluding batch size).
        Example: (122,), (32, 32, 3)

    Returns
    -------
    dict
        Total Parameters
        Trainable Parameters
        Non-trainable Parameters
        Model Size (MB)
        FLOPs (GFLOPs)
    """

    # Build model if not already built
    model.build((None, *input_shape))

    # -----------------------------
    # Parameters
    # -----------------------------
    total_params = model.count_params()

    trainable_params = np.sum(
        [tf.keras.backend.count_params(w) for w in model.trainable_weights]
    )

    non_trainable_params = np.sum(
        [tf.keras.backend.count_params(w) for w in model.non_trainable_weights]
    )

    # -----------------------------
    # Model Size
    # -----------------------------
    model_size_bytes = total_params * 4       # float32 = 4 bytes
    model_size_mb = model_size_bytes / (1024 ** 2)

    # -----------------------------
    # FLOPs
    # -----------------------------
    @tf.function
    def forward_pass(x):
        return model(x)

    concrete_func = forward_pass.get_concrete_function(
        tf.TensorSpec([1, *input_shape], tf.float32)
    )

    from tensorflow.python.framework.convert_to_constants import (
        convert_variables_to_constants_v2,
    )
    
    frozen_func = convert_variables_to_constants_v2(concrete_func)

    graph_def = frozen_func.graph.as_graph_def()

    with tf.Graph().as_default() as graph:
        tf.graph_util.import_graph_def(graph_def, name="")

        run_meta = tf.compat.v1.RunMetadata()

        opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()

        flops = tf.compat.v1.profiler.profile(
            graph=graph,
            run_meta=run_meta,
            cmd='op',
            options=opts
        )

    total_flops = flops.total_float_ops
    gflops = total_flops / 1e9

    return {
        "Total Parameters": int(total_params),
        "Trainable Parameters": int(trainable_params),
        "Non-trainable Parameters": int(non_trainable_params),
        "Model Size (MB)": round(model_size_mb, 3),
        "FLOPs": int(total_flops),
        "GFLOPs": round(gflops, 4)
    }

In [ ]:
import matplotlib.pyplot as plt

def plot_accuracy(history, save_path=None):
    """
    Plot Training vs Validation Accuracy.

    Parameters
    ----------
    history : keras.callbacks.History
        History object returned by model.fit().
    save_path : str, optional
        Path to save the figure.
    """

    plt.figure(figsize=(7, 5))

    plt.plot(
        history.history['accuracy'],
        label='Training Accuracy',
        linewidth=2,
        marker='o',
        markersize=4
    )

    plt.plot(
        history.history['val_accuracy'],
        label='Validation Accuracy',
        linewidth=2,
        marker='s',
        markersize=4
    )

    plt.title('Training vs Validation Accuracy', fontsize=14)
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Accuracy', fontsize=12)

    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(loc='lower right')

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')

    plt.show()


import matplotlib.pyplot as plt

def plot_loss(history, save_path=None):
    """
    Plot Training vs Validation Loss.

    Parameters
    ----------
    history : keras.callbacks.History
        History object returned by model.fit().
    save_path : str, optional
        Path to save the figure.
    """

    plt.figure(figsize=(7, 5))

    plt.plot(
        history.history['loss'],
        label='Training Loss',
        linewidth=2,
        marker='o',
        markersize=4
    )

    plt.plot(
        history.history['val_loss'],
        label='Validation Loss',
        linewidth=2,
        marker='s',
        markersize=4
    )

    plt.title('Training vs Validation Loss', fontsize=14)
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Loss', fontsize=12)

    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(loc='upper right')

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')

    plt.show()

    

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    roc_curve,
    roc_auc_score
)
from sklearn.preprocessing import label_binarize
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


def plot_confusion_matrix(
    y_true,
    y_pred,
    class_names,
    title,
    save_path=None,
    normalize=False
):

    cm = confusion_matrix(y_true, y_pred)

    if normalize:
        cm = cm.astype(float) / (
            cm.sum(axis=1)[:, np.newaxis] + 1e-10
        )

    plt.figure(figsize=(8,6))

    sns.heatmap(
        cm,
        annot=True,
        fmt=".2f" if normalize else "d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names
    )

    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=700)

    plt.show()
    plt.close()



def plot_multiclass_roc(
    y_true,
    y_prob,
    class_names,
    title,
    save_path=None
):

    y_true = np.asarray(y_true).ravel()
    y_prob = np.asarray(y_prob)

    plt.figure(figsize=(7, 6))

    # --------------------------------------------------
    # Binary classification
    # --------------------------------------------------

    if y_prob.shape[1] == 2:

        # Probability of positive class (1)
        y_score = y_prob[:, 1]

        # ROC curve
        fpr, tpr, _ = roc_curve(
            y_true,
            y_score
        )

        auc_score = roc_auc_score(
            y_true,
            y_score
        )

        plt.plot(
            fpr,
            tpr,
            label=f"ROC (AUC = {auc_score:.4f})"
        )

    # --------------------------------------------------
    # Multiclass classification
    # --------------------------------------------------

    else:

        y_true_bin = label_binarize(
            y_true,
            classes=np.arange(len(class_names))
        )

        auc_score = roc_auc_score(
            y_true_bin,
            y_prob,
            multi_class="ovr",
            average="macro"
        )

        for i, class_name in enumerate(class_names):

            fpr, tpr, _ = roc_curve(
                y_true_bin[:, i],
                y_prob[:, i]
            )

            class_auc = roc_auc_score(
                y_true_bin[:, i],
                y_prob[:, i]
            )

            plt.plot(
                fpr,
                tpr,
                label=f"{class_name} (AUC = {class_auc:.4f})"
            )

    # --------------------------------------------------
    # Random classifier
    # --------------------------------------------------

    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        label="Random"
    )

    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")

    plt.title(title)

    plt.legend()
    plt.grid(True)

    if save_path is not None:
        plt.savefig(
            save_path,
            dpi=300,
            bbox_inches="tight"
        )

    plt.show()

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    roc_auc_score,
    matthews_corrcoef,
    confusion_matrix
)
from sklearn.preprocessing import label_binarize

def classwise_metrics(y_true, y_pred, y_prob, class_names):

    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    y_prob = np.asarray(y_prob)

    classes = np.array([0, 1])

    results = []

    for cls in classes:

        # One-vs-rest labels
        yt = (y_true == cls).astype(int)
        yp = (y_pred == cls).astype(int)

        # Class probability
        if cls == 0:
            y_score = y_prob[:, 0]
        else:
            y_score = y_prob[:, 1]

        # Confusion matrix
        tn, fp, fn, tp = confusion_matrix(
            yt,
            yp,
            labels=[0, 1]
        ).ravel()

        # Precision
        precision = (
            tp / (tp + fp)
            if (tp + fp) > 0
            else 0
        )

        # Recall / sensitivity
        recall = (
            tp / (tp + fn)
            if (tp + fn) > 0
            else 0
        )

        # Specificity
        specificity = (
            tn / (tn + fp)
            if (tn + fp) > 0
            else 0
        )

        # F1
        f1 = (
            2 * precision * recall / (precision + recall)
            if (precision + recall) > 0
            else 0
        )

        # FPR
        fpr = (
            fp / (fp + tn)
            if (fp + tn) > 0
            else 0
        )

        # G-Mean
        gmean = np.sqrt(
            recall * specificity
        )

        # MCC
        denominator = np.sqrt(
            (tp + fp) *
            (tp + fn) *
            (tn + fp) *
            (tn + fn)
        )

        mcc = (
            ((tp * tn) - (fp * fn)) / denominator
            if denominator > 0
            else 0
        )

        # ROC-AUC
        if len(np.unique(yt)) == 2:
            roc_auc = roc_auc_score(
                yt,
                y_score
            )
        else:
            roc_auc = np.nan

        # Support
        support = np.sum(yt)

        results.append({
            "Class": class_names[cls],
            "Precision": precision,
            "Recall": recall,
            "F1": f1,
            "FPR": fpr,
            "GMean": gmean,
            "MCC": mcc,
            "ROC_AUC": roc_auc,
            "Support": support
        })

    return pd.DataFrame(results)
def calculate_metrics(
    y_true,
    y_pred,
    y_prob
):

    # Calculate multiclass FPR using One-vs-Rest
    classes = np.unique(y_true)

    fprs = []
    supports = []

    for cls in classes:

        # One-vs-Rest transformation
        yt = (np.array(y_true) == cls).astype(int)
        yp = (np.array(y_pred) == cls).astype(int)

        tn, fp, fn, tp = confusion_matrix(yt, yp).ravel()

        # False Positive Rate
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0

        fprs.append(fpr)
        supports.append(np.sum(yt))

    # Weighted FPR
    weighted_fpr = np.average(
        fprs,
        weights=supports
    )

    metrics = {

        "Accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),
    
        "Precision":
            precision_score(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0
            ),
    
        "Recall":
            recall_score(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0
            ),
    
        "F1":
            f1_score(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0
            ),
    
        "FPR":
            weighted_fpr,
    
        "ROC_AUC":
            roc_auc_score(
                y_true,
                y_prob[:, 1]
            ),
    
        "GMean":
            geometric_mean_score(
                y_true,
                y_pred
            ),
    
        "MCC":
            matthews_corrcoef(
                y_true,
                y_pred
            )
    }
    return pd.DataFrame([metrics])
def timing_report(
    train_time,
    test_time,
    predict_time
):

    return pd.DataFrame({

        "Training Time (s)":
            [train_time],

        "Testing Time (s)":
            [test_time],

        "Prediction Time (s)":
            [predict_time],

        "Total Time (s)":
            [
                train_time +
                test_time +
                predict_time
            ]

    })
import tensorflow as tf
import numpy as np


In [ ]:
def evaluate_dataset(
    model,
    X,
    y,
    dataset_name,
    class_names
):

    print(f"\n{'='*60}")
    print(dataset_name.upper())
    print(f"{'='*60}")

    start = time.time()

    y_prob = model.predict(X, verbose=0)

    # Binary classification
    # y_prob shape = (n_samples, 2)
    y_score = y_prob[:, 1]

    # Convert probability of class 1 to binary prediction
    y_pred = (y_score >= 0.5).astype(int)

    pred_time = time.time() - start

    metrics_df = calculate_metrics(
        y,
        y_pred,
        y_prob
    )

    display(metrics_df.round(4))

    class_df = classwise_metrics(
        y,
        y_pred,
        y_prob,
        class_names
    )

    print("\nClass-wise Performance")
    display(class_df)

    plot_confusion_matrix(
        y,
        y_pred,
        class_names,
        f"{dataset_name} Confusion Matrix"
    )

    plot_confusion_matrix(
        y,
        y_pred,
        class_names,
        f"{dataset_name} Normalized CM",
        normalize=True
    )

    plot_multiclass_roc(
        y,
        y_prob,
        class_names,
        f"{dataset_name} ROC Curve"
    )

    return metrics_df, class_df, pred_time

In [ ]:

CLASS_NAMES =list(set(y_train))
NUM_CLASSES =len(set(CLASS_NAMES))

In [ ]:
import subprocess


def get_gpu_usage():

    try:
        result = subprocess.check_output(
            [
                "nvidia-smi",
                "--query-gpu=utilization.gpu",
                "--format=csv,noheader,nounits"
            ],
            stderr=subprocess.DEVNULL
        ).decode("utf-8").strip()

        if not result:
            return 0.0

        # If multiple GPUs exist, return average utilization
        values = [
            float(x.strip())
            for x in result.splitlines()
            if x.strip()
        ]

        return sum(values) / len(values)

    except (subprocess.CalledProcessError, FileNotFoundError, ValueError):
        return 0.0

In [ ]:
from keras.regularizers import l2
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

checkpoint = ModelCheckpoint(
    "best_model.keras",
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

In [ ]:
gpu_before_model = get_gpu_usage()
model = create_model(
    input_shape=input_shape,
    lstm_units=16,
    l2_reg=1e-4,
    dense1=32,
    dense2=16,
    n_classes=n_classes
)

print("Model Complexity Analysis")
metrics = model_complexity(
    model,
    input_shape=input_shape     # Example input shape
)

metrics_df = pd.DataFrame(
    list(metrics.items()),
    columns=["Metric", "Value"]
)

metrics_df["Value"] = pd.to_numeric(
    metrics_df["Value"], errors="ignore"
)

display(metrics_df.round(4))
    
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
gpu_after_model = get_gpu_usage()
mem_after_model = memory_usage_mb()
print(f"Memory after model build: {mem_after_model:.2f} MB")
# =====================================================
# TRAIN MODEL
# =====================================================
start_train = time.time()
print("Training model...")

history = model.fit(
    X_train_reshaped,
    y_train,
    validation_data=(X_val_reshaped, y_val),
    epochs=100,
    batch_size=256,
    callbacks=[
        early_stopping,
        reduce_lr,
        checkpoint
    ],
    verbose=1
)

plot_accuracy(history)
plot_loss(history)


end_train = time.time()
gpu_after_train = get_gpu_usage()
mem_after_train = memory_usage_mb()
print(f"Memory after training: {mem_after_train:.2f} MB")
train_time = end_train - start_train
print(f"Training time: {train_time:.2f}s")

# =====================================================
# TEST EVALUATION
# =====================================================
start_test = time.time()
loss, accuracy = model.evaluate(X_test_reshaped, y_test)
end_test = time.time()


gpu_after_test = get_gpu_usage()
mem_after_test = memory_usage_mb()
print(f"Memory after test data evaluation: {mem_after_test:.2f} MB")
test_time = end_test - start_test

# =====================================================
# VALIDATION EVALUATION
# =====================================================
start_val = time.time()
val_loss, val_accuracy = model.evaluate(X_val_reshaped, y_val)
end_val = time.time()


gpu_after_val = get_gpu_usage()
mem_after_val = memory_usage_mb()
print(f"Memory after validation data evaluation: {mem_after_val:.2f} MB")
val_time = end_val - start_val


In [ ]:
# =====================================================
# PREDICTION
# =====================================================
start_pred = time.time()
predictions = model.predict(X_test_reshaped)

predicted_classes = np.argmax(predictions,axis=1)
true_classes = y_test

precision = precision_score(true_classes, predicted_classes,average='binary')
recall = recall_score(true_classes, predicted_classes,average='binary')
f1 = f1_score(true_classes, predicted_classes,average='macro')

end_pred = time.time()
mem_end = memory_usage_mb()
gpu_end = get_gpu_usage()
print(f"Memory after prediction: {mem_end:.2f} MB")
predict_time = end_pred - start_pred

In [ ]:
import pandas as pd
from IPython.display import display

# ------------------ MEMORY SUMMARY ------------------

memory_summary = pd.DataFrame({
    "Stage": [
        "Start",
        "After data",
        "After model",
        "After train",
        "After eval",
        "After val",
        "After predict"
    ],
    "Memory Usage (MB)": [
        mem_start,
        mem_after_data,
        mem_after_model,
        mem_after_train,
        mem_after_test,
        mem_after_val,
        mem_end
    ]
})

# ------------------ GPU SUMMARY ------------------
gpu_summary = pd.DataFrame({
    "Stage": [
        "Before model",
        "After model",
        "After train",
        "After eval",
        "After val",
        "After predict",
      
    ],
    "Memory Usage (MB)": [
        gpu_before_model,
        gpu_after_model,
        gpu_after_train,
        gpu_after_test,
        gpu_after_val,
        gpu_end
    ]
})

# ------------------ TIME SUMMARY ------------------

time_summary = pd.DataFrame({
    "Stage": [
        "Train",
        "Eval",
        "Val",
        "Predict",
        "Total"
    ],
    "Time (s)": [
        train_time,
        test_time,
        val_time,
        predict_time,
        train_time + test_time + val_time
    ]
})

print("Memory Usage Summary")
display(memory_summary.round(2))

print("GPU Usage Summary")
display(gpu_summary.round(2))


print("\nTime Summary")
display(time_summary.round(2))

In [ ]:
dataset_name='IoT23'

In [ ]:
train_metrics, train_class_metrics, train_pred_time = evaluate_dataset(
    model,
    X_train_reshaped,
    y_train,
    "Training",
    CLASS_NAMES
)

In [ ]:
val_metrics, val_class_metrics, val_pred_time = evaluate_dataset(
    model,
    X_val_reshaped,
    y_val,
    "Validation",
    CLASS_NAMES
)

In [ ]:
test_metrics, test_class_metrics, test_pred_time = evaluate_dataset(
    model,
    X_test_reshaped,
    y_test,
    "Testing",
    CLASS_NAMES
)

In [ ]:

print("*"*50)
print("Performance Summary")
print("*"*50)
summary = pd.concat(

    [
        train_metrics.assign(Dataset="Train"),
        val_metrics.assign(Dataset="Validation"),
        test_metrics.assign(Dataset="Test")
    ]

)

summary = summary.set_index("Dataset")
display(summary.round(4))

In [ ]:
"done"